# B2 · Tokenisation et embeddings — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b2_tokenisation/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B2.Q1 à B2.Q12, B2.R1 à B2.R3, B2.1 à B2.6, B2.9 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★ | |
| A | B2.13 à B2.14 | des tokenizers réels sur Holmes et Verne, et ce que coûtent les langues, les nombres et les emoji (*prochaine session de génération*) | | | |
| B | B2.15 à B2.19 | ton BPE : compter et fusionner les paires, apprendre les fusions, encoder et décoder, réparer un tokenizer qui abîme les accents, te comparer à la bibliothèque tokenizers (*prochaine session de génération*) | | | |
| C | B2.20 à B2.21 | expériences : compression et entropie selon la taille du vocabulaire, tokenizer d'une langue appliqué à l'autre (*prochaine session de génération*) | | | |
| D | B2.22 à B2.27 | vecteurs de mots : co-occurrences et PPMI, cosinus et voisins, PPMI + SVD sur Holmes, word2vec sur Verne, voisins de « Fogg » et « Passepartout », carte des embeddings (*prochaine session de génération*) | | | |
| E | B2.28 à B2.31 | embeddings de phrases et recherche sémantique, statique contre contextuel, sauvegarder un tokenizer, et le défi (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.18 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Manipuler Unicode, UTF-8 et la normalisation d'un texte, et compter ses octets et ses points de code.
- Écrire un tokenizer BPE (entraînement, encodage par rang, décodage) et le comparer à la bibliothèque `tokenizers`.
- Comparer des tokenizers réels et mesurer leur coût (fertilité, compression) selon la langue.
- Relier `nn.Embedding` à un produit one-hot × table, et entraîner des embeddings word2vec.
- Construire des embeddings par co-occurrences (PPMI et SVD), les interroger par le cosinus, et chercher des passages par des embeddings de phrases.

**Rappel express.** Fertilité $= \frac{\text{tokens}}{\text{mots}}$, compression $= \frac{\text{octets}}{\text{tokens}}$. Fusion BPE : la paire voisine la plus fréquente (à égalité, la plus petite paire d'identifiants) devient un token ; on encode en appliquant les fusions par rang. $\mathbf{x}_{\text{one-hot}}^\top \mathbf{E} = \mathbf{E}[i, :]$. $\operatorname{PPMI}(w, c) = \max\big(0, \log \frac{p(w, c)}{p(w)\,p(c)}\big)$. Skip-gram : $L = -\log \sigma(\mathbf{u}_c^\top \mathbf{v}_w) - \sum_k \log \sigma(-\mathbf{u}_{n_k}^\top \mathbf{v}_w)$. Mean pooling : $\mathbf{s} = \frac{\sum_t m_t \mathbf{h}_t}{\sum_t m_t}$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B2.1 à B2.6, 🧮 B2.9)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"B"`) ; une forme de tenseur : entre guillemets (`"(2, 3)"`) ; un mot : entre guillemets (`"mot"`) ; des octets : une liste, en hexadécimal ou en décimal (`[0x4E, 0x6F]` ou `[78, 111]`) ; des identifiants, une paire d'identifiants ou un vecteur : une liste (`[4, 0, 7]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la dérivation ∂ B2.7, l'estimation 🧮 B2.8, la réflexion (🗣️ B2.10, ⚖️ B2.11, 📄 B2.12) et l'entretien se corrigent avec `05_solutions.md`.

In [2]:
# The short answers only use the numbers of their statements (02_exercices.md)
import math
import unicodedata

import numpy as np

# 🧠 Q2: « é » in NFC (one code point) and in NFD (« e » + U+0301)
E_ACUTE_NFC = "\u00e9"
E_ACUTE_NFD = unicodedata.normalize("NFD", E_ACUTE_NFC)

# ✏️ B2.1: « Noël à Paris », typed in NFC
S_B21 = "No\u00ebl \u00e0 Paris"
B_B21 = S_B21.encode("utf-8")
NFD_B21 = unicodedata.normalize("NFD", S_B21)


def utf8_two_bytes(code_point):
    """110xxxxx 10xxxxxx: the 5 high bits, then the 6 low bits of an 11-bit code point (U+0080 to U+07FF)."""
    return [0b11000000 | (code_point >> 6), 0b10000000 | (code_point & 0b111111)]


# ✏️ B2.2: toy vectors (axes: royalty, masculine (+) or feminine (-), adult), « homme est à roi ce que femme est à ? »
WORDS_B22 = ["roi", "reine", "homme", "femme", "prince", "princesse", "trône"]
E_B22 = np.array([[0.6, 1.0, 0.9], [0.8, -0.5, 0.9], [0.1, 1.0, 0.9], [0.1, -1.0, 0.9],
                  [0.7, 0.8, 0.2], [0.7, -0.6, 0.2], [3.0, 0.0, 0.5]])
Q_B22 = E_B22[0] - E_B22[2] + E_B22[3]
COS_B22 = E_B22 @ Q_B22 / (np.linalg.norm(E_B22, axis=1) * np.linalg.norm(Q_B22))
E_UNIT_B22 = E_B22 / np.linalg.norm(E_B22, axis=1, keepdims=True)


def analogy_b22(E, a, b, c, exclude=True, cosine=True):
    """The word x of largest cos(x, b - a + c) (or dot product), without a, b and c when exclude is True."""
    k = {w: i for i, w in enumerate(WORDS_B22)}
    q = E[k[b]] - E[k[a]] + E[k[c]]
    scores = E @ q / (np.linalg.norm(E, axis=1) * np.linalg.norm(q)) if cosine else E @ q
    order = [i for i in np.argsort(-scores, kind="stable") if not (exclude and WORDS_B22[i] in (a, b, c))]
    return WORDS_B22[order[0]]


# ✏️ B2.3: three BPE merges by hand; ties go to the smallest pair of ids (BpeTrainer of tokenizers)
CORPUS_B23 = {"été": 4, "étés": 3, "pré": 2, "prés": 1}


def merge_ids(ids, pair, new_id):
    """Replace each occurrence of pair, left to right and without overlap, by new_id."""
    out, i = [], 0
    while i < len(ids):
        if i + 1 < len(ids) and (ids[i], ids[i + 1]) == pair:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out


def learn_bpe(corpus, n_merges):
    """The fiche's pseudo-code: alphabet sorted by code point, then n_merges merges (pair of ids) with their counts."""
    alphabet = sorted({ch for word in corpus for ch in word})
    words = {tuple(alphabet.index(ch) for ch in word): f for word, f in corpus.items()}
    merges, counts = [], []
    for _ in range(n_merges):
        pairs = {}
        for word, f in words.items():
            for pair in zip(word, word[1:]):
                pairs[pair] = pairs.get(pair, 0) + f
        best = min(pairs, key=lambda p: (-pairs[p], p))           # most frequent, then the smallest pair of ids
        merges.append(best)
        counts.append(pairs[best])
        words = {tuple(merge_ids(list(w), best, len(alphabet) + len(merges) - 1)): f for w, f in words.items()}
    return alphabet, merges, counts, words


ALPHABET_B23, MERGES_B23, COUNTS_B23, WORDS_AFTER_B23 = learn_bpe(CORPUS_B23, 3)
TOKENS_AFTER_B23 = sum(len(word) * f for word, f in WORDS_AFTER_B23.items())

# ✏️ B2.4: a BPE already learned (char level), applied by rank; an unseen character becomes <unk>
ALPHABET_B24 = list("acehnorst")
MERGES_B24 = [("n", "t"), ("c", "h"), ("a", "nt"), ("o", "n"), ("ch", "ant"), ("e", "r"), ("on", "s"),
              ("chant", "er")]
VOCAB_B24 = {ch: i for i, ch in enumerate(ALPHABET_B24)}
for _a, _b in MERGES_B24:
    VOCAB_B24[_a + _b] = len(VOCAB_B24)
UNK_B24 = len(VOCAB_B24)                                             # the special token comes after the merges
RANK_B24 = {(VOCAB_B24[a], VOCAB_B24[b]): (r, VOCAB_B24[a + b]) for r, (a, b) in enumerate(MERGES_B24)}


def encode_b24(word, by_rank=True):
    """Merge the adjacent pair of lowest rank (by_rank) or, the classic mistake, the leftmost known pair."""
    ids = [VOCAB_B24.get(ch, UNK_B24) for ch in word]
    while True:
        present = [(RANK_B24[p][0], i) for i, p in enumerate(zip(ids, ids[1:])) if p in RANK_B24]
        if not present:
            return ids
        _, i = min(present) if by_rank else min(present, key=lambda t: t[1])
        ids[i:i + 2] = [RANK_B24[(ids[i], ids[i + 1])][1]]


# ✏️ B2.5: one-hot times a table, and the gradient of L = sum over the positions t of w . e_t
E_B25 = np.array([[1, 0, 2], [0, 1, -1], [2, -1, 0], [-1, 3, 1], [0, 2, 2]], dtype=float)
IDS_B25 = [3, 0, 3, 1]
W_B25 = np.array([1.0, -2.0, 0.5])
X_B25 = np.eye(5)[IDS_B25]                                           # one one-hot row per position: (4, 5)
OUT_B25 = X_B25 @ E_B25                                              # (4, 3)
L_B25 = float((OUT_B25 @ W_B25).sum())
GRAD_B25 = X_B25.T @ np.tile(W_B25, (len(IDS_B25), 1))               # dL/dE = X^T dL/dOUT, shape (5, 3)
LR_B25 = 0.1

# ✏️ B2.6: co-occurrences in a window of 1, ids in alphabetical order (chat 0, court 1, le 2, rat 3)
WORDS_B26 = ["chat", "court", "le", "rat"]
IDS_B26 = [WORDS_B26.index(w) for w in "le chat court le chat le rat court".split()]
C_B26 = np.zeros((4, 4))
for _i in range(len(IDS_B26) - 1):
    C_B26[IDS_B26[_i], IDS_B26[_i + 1]] += 1                         # right context
    C_B26[IDS_B26[_i + 1], IDS_B26[_i]] += 1                         # left context
N_B26 = C_B26.sum()


def pmi_b26(w, c, alpha=1.0, alpha_rows=False):
    """log p(w, c) / (p(w) p_alpha(c)); alpha_rows smooths the rows too (a classic mistake)."""
    i, j = WORDS_B26.index(w), WORDS_B26.index(c)
    rows = C_B26.sum(axis=1) ** (alpha if alpha_rows else 1.0)
    cols = C_B26.sum(axis=0) ** alpha
    return math.log((C_B26[i, j] / N_B26) / ((rows[i] / rows.sum()) * (cols[j] / cols.sum())))


P_ALPHA_B26 = C_B26.sum(axis=0) ** 0.75 / (C_B26.sum(axis=0) ** 0.75).sum()

# 🧮 B2.9: embedding tables (config.json and safetensors headers of the Hugging Face Hub, read on 2026-11-01)
GPT2_TABLE_B29 = 50257 * 768
GPT2_TOTAL_B29 = 124_439_808                                         # GPT-2 small, its tied head counted once
L31_TABLE_B29 = 128256 * 4096                                        # Llama 3.1 8B, untied: two tables
L31_TOTAL_B29 = 8_030_261_248
L32_TABLE_B29 = 128256 * 2048                                        # Llama 3.2 1B, tied: one table
L32_TOTAL_B29 = 1_235_814_400

# quizzes and recalls with a computation
WP_AB_Q7 = 20 / (40 * 100)                                          # WordPiece score count(a, b) / (count(a) count(b))
WP_CD_Q7 = 30 / (300 * 200)
PMI_Q10 = math.log(6 * 1000 / (30 * 200))                           # p(w, c) / (p(w) p(c)), N simplified
H_Q11 = np.array([[1.0, 2.0], [3.0, 0.0], [2.0, 4.0], [0.0, 0.0]])   # the last position is a [PAD]
M_Q11 = np.array([1.0, 1.0, 1.0, 0.0])
S_Q11 = (M_Q11[:, None] * H_Q11).sum(axis=0) / M_Q11.sum()
P_R3 = np.array([0.5, 0.25, 0.125, 0.125])
H_R3 = float(-(P_R3 * np.log2(P_R3)).sum())

# The same answers from the libraries (no download): mylearn (reference), tokenizers, PyTorch
import torch
from tokenizers import Tokenizer, models, pre_tokenizers, trainers
from torch import nn

assert len(E_ACUTE_NFC.encode()) == 2 and len(E_ACUTE_NFD) == 2 and len(E_ACUTE_NFD.encode()) == 3
assert len("\u2019".encode()) == 3 and len("\u4e2d".encode()) == 3
assert utf8_two_bytes(0xEB) == list("\u00eb".encode()) and utf8_two_bytes(0x153) == list("\u0153".encode())

T = mylearn.tokenizer
# ✏️ B2.3: the merges of BpeTrainer and of BPETokenizer, char level, Whitespace pre-tokenizer
text_b23 = " ".join(w for w, f in CORPUS_B23.items() for _ in range(f))
ref_b23 = T.BPETokenizer(vocab_size=len(ALPHABET_B23) + 3, level="char", min_frequency=1).fit(text_b23)
assert ref_b23.merges_ == MERGES_B23 and ref_b23.n_base_ == len(ALPHABET_B23)
hf_b23 = Tokenizer(models.BPE())
hf_b23.pre_tokenizer = pre_tokenizers.Whitespace()
hf_b23.train_from_iterator([text_b23], trainers.BpeTrainer(vocab_size=len(ALPHABET_B23) + 3, min_frequency=1,
                                                           show_progress=False))
assert [hf_b23.token_to_id(ch) for ch in ALPHABET_B23] == list(range(len(ALPHABET_B23)))
assert [hf_b23.token_to_id(ref_b23.vocab_[a].decode() + ref_b23.vocab_[b].decode()) for a, b in MERGES_B23] == [5, 6, 7]
for min_frequency, n in ((3, 3), (4, 2)):                             # 🧠 B2.3 h): the stopping rule
    assert len(T.BPETokenizer(vocab_size=8, level="char", min_frequency=min_frequency).fit(text_b23).merges_) == n

# ✏️ B2.4: the same merges in models.BPE and in BPETokenizer (attributes set by hand)
hf_b24 = Tokenizer(models.BPE(vocab=dict(VOCAB_B24, **{"<unk>": UNK_B24}), merges=MERGES_B24, unk_token="<unk>"))
ref_b24 = T.BPETokenizer(level="char", unk_token="<unk>")
ref_b24.n_base_, ref_b24.special_ids_ = len(ALPHABET_B24), {"<unk>": UNK_B24}
ref_b24.merges_ = [(VOCAB_B24[a], VOCAB_B24[b]) for a, b in MERGES_B24]
ref_b24.vocab_ = {i: s.encode() for s, i in VOCAB_B24.items()} | {UNK_B24: b"<unk>"}
for word in ("chantons", "ont", "chanteront", "chantez", "enchanter", "chantonner"):
    assert encode_b24(word) == hf_b24.encode(word).ids == ref_b24.encode(word), word
assert encode_b24("ont", by_rank=False) != encode_b24("ont")
byte_b24 = T.BPETokenizer(vocab_size=256, pre_tokenizer="gpt2").fit("x")       # 256 bytes, no merge
assert len(byte_b24.encode("été")) == 5 and ref_b24.decode(ref_b24.encode("chantez")) != "chantez"

# ✏️ B2.5: nn.Embedding, its gradient, padding_idx and the weight decay of SGD
emb = nn.Embedding(5, 3)
with torch.no_grad():
    emb.weight.copy_(torch.tensor(E_B25))
ids = torch.tensor(IDS_B25)
x = nn.functional.one_hot(ids, num_classes=5).double()
assert torch.allclose(x @ emb.weight.double(), emb(ids).double()) and np.allclose(emb(ids).detach(), OUT_B25)
loss = (emb(ids) @ torch.tensor(W_B25, dtype=torch.float32)).sum()
loss.backward()
assert np.isclose(loss.item(), L_B25) and np.allclose(emb.weight.grad, GRAD_B25)
torch.optim.SGD(emb.parameters(), lr=LR_B25).step()
assert np.allclose(emb.weight.detach()[3], E_B25[3] - LR_B25 * GRAD_B25[3])
pad = nn.Embedding(5, 3, padding_idx=0)
(pad(ids) @ torch.tensor(W_B25, dtype=torch.float32)).sum().backward()
assert torch.all(pad.weight.grad[0] == 0)
decay = nn.Embedding(5, 3)
before = decay.weight.detach().clone()
(decay(ids) @ torch.tensor(W_B25, dtype=torch.float32)).sum().backward()
torch.optim.SGD(decay.parameters(), lr=LR_B25, weight_decay=0.01).step()
assert not torch.equal(decay.weight.detach()[4], before[4])

# ✏️ B2.6: cooccurrence_matrix and ppmi of the reference
Em = mylearn.embeddings
assert np.array_equal(Em.cooccurrence_matrix(IDS_B26, 4, window=1), C_B26)
P1, P75 = Em.ppmi(C_B26), Em.ppmi(C_B26, alpha=0.75)
assert np.isclose(P1[0, 2], pmi_b26("chat", "le")) and P1[1, 2] == 0 and pmi_b26("court", "le") < 0
assert np.isclose(P75[0, 2], pmi_b26("chat", "le", 0.75)) and P75[1, 2] > 0 and not np.allclose(P75, P75.T)

# ✏️ B2.2: most_similar of the reference, with and without the exclusion
i22 = {w: i for i, w in enumerate(WORDS_B22)}
assert WORDS_B22[Em.most_similar(E_B22, Q_B22, k=1)[0][0]] == analogy_b22(E_B22, "homme", "roi", "femme", exclude=False)
assert WORDS_B22[Em.most_similar(E_B22, Q_B22, k=1, exclude=(i22["homme"], i22["roi"], i22["femme"]))[0][0]] == "reine"

# 🧮 B2.9: GPT-2 small built without its weights (meta device)
from transformers import GPT2Config, GPT2LMHeadModel
with torch.device("meta"):
    gpt2 = GPT2LMHeadModel(GPT2Config())
assert sum(p.numel() for p in gpt2.parameters()) == GPT2_TOTAL_B29
assert gpt2.transformer.wte.weight.numel() == GPT2_TABLE_B29 and gpt2.lm_head.weight is gpt2.transformer.wte.weight

# 🔁 R2 and R3: build_vocab, encode (ch. 22), entropy and Huffman (ch. 6)
S = mylearn.sequences
stoi, itos = S.build_vocab("banana")
assert stoi["b"] == 2
stoi_u, _ = S.build_vocab("banana", unk_token="<unk>")
assert S.encode("bandana", stoi_u, unk_token="<unk>").tolist() == [3, 1, 2, 0, 1, 2, 1]
stoi_w, itos_w = S.build_vocab(["le", "chat", "le", "rat"], max_size=2, unk_token="<unk>")
assert S.encode(["chat"], stoi_w, unk_token="<unk>").tolist() == [0]
assert np.isclose(mylearn.info.entropy(P_R3), H_R3)
assert min(len(c) for c in mylearn.info.huffman_code(list("abcd"), P_R3).values()) == 1
print("✅ the paper answers agree with mylearn (reference), tokenizers and PyTorch")

/root/venv313/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ the paper answers agree with mylearn (reference), tokenizers and PyTorch


**B2.Q1 — Caractères, mots ou sous-mots : taille du vocabulaire contre longueur des séquences**

In [3]:
wb.record("B2.Q1a", "A", mistakes={"un mot entier est un seul token : relis la colonne « longueur des séquences »": "B", "les sous-mots sont entre les deux : relis le tableau des trois granularités": "C"})
wb.record("B2.Q1b", True, mistakes={"relis ce que devient un mot qui n'est pas parmi les V plus fréquents (§202.1)": False})
wb.record("B2.Q1c", 3 ** 2, mistakes={"le coût croît comme le carré de la longueur, pas comme la longueur": 3,
          "le carré de 3, pas son double": 6,
          "le carré de la longueur, pas son cube": 27})
wb.record("B2.Q1d", "B", mistakes={"personne ne choisit les morceaux à la main : relis d'où vient le découpage": "A", "les morceaux n'ont pas de longueur fixe : relis l'exemple de « paisiblement »": "C"})
wb.record("B2.Q1e", False, mistakes={"relis le tableau des trois granularités : combien de symboles distincts à chaque niveau ?": True})

WB_ANSWER {"hash": "e409042f12dbb0a0c98cc8712cac692ca9359e061d70b21ec5017d3fc60d7a53", "id": "B2.Q1a", "kind": "str", "mistakes": {"34484e9283426b8754ddd4418545f93684e57b11a84243e68226a415260b7f55": "un mot entier est un seul token : relis la colonne « longueur des séquences »", "4e1c8a7750c65060cfaf8677b0a93ce23f42ab629ccf25d840e8d0d109ac45a0": "les sous-mots sont entre les deux : relis le tableau des trois granularités"}}
📝 Ex B2.Q1a : réponse enregistrée (str).
WB_ANSWER {"hash": "3bfa7f40abe1dd42f341d809889c22795d2c342ce705490a2b2bcb071a61398b", "id": "B2.Q1b", "kind": "bool", "mistakes": {"d118f5223c7b349d78b5ecf8315c0e177211d9638861b630ad1536efca2f286c": "relis ce que devient un mot qui n'est pas parmi les V plus fréquents (§202.1)"}}
📝 Ex B2.Q1b : réponse enregistrée (bool).
WB_ANSWER {"hash": "2231eb484c560f65199aa2062fc3e6b27cd664709be63d4cbb56f5c04afb3369", "id": "B2.Q1c", "kind": "int", "magnitude": 0, "mistakes": {"36201ba999407ce8330ad80f2c00d65f8c6d090887ffe83b28bc01edbc2

**B2.Q2 — Octets, points de code, caractères affichés : combien « é » compte-t-il ?**

In [4]:
wb.record("B2.Q2a", len(E_ACUTE_NFC.encode('utf-8')), mistakes={"« é » n'est pas un caractère ASCII : relis le tableau d'UTF-8": 1,
          "U+00E9 est sous U+0800 : relis les plages du tableau d'UTF-8": 3})
wb.record("B2.Q2b", len(E_ACUTE_NFD), mistakes={"relis ce que fait NFD d'une lettre accentuée": 1,
          "on compte ici des points de code, pas des octets": 3})
wb.record("B2.Q2c", len(E_ACUTE_NFD.encode('utf-8')), mistakes={"compte les octets de chacun des deux points de code": 2,
          "« e » est un caractère ASCII : un seul octet": 4})
wb.record("B2.Q2d", len(E_ACUTE_NFC), mistakes={"len d'un str compte les points de code, pas les octets": 2})
wb.record("B2.Q2e", False, mistakes={"U+2019 est au-delà de U+07FF : relis les plages du tableau d'UTF-8": True})

WB_ANSWER {"hash": "4f2a13b560d701a209d6a607436f81cd4b9d0c853fdba462da8a2f56994e5dd9", "id": "B2.Q2a", "kind": "int", "magnitude": 0, "mistakes": {"65cee967f07554e15555e0f5b2d61394c2074e8201ee52b988a3022f0d91fb2d": "U+00E9 est sous U+0800 : relis les plages du tableau d'UTF-8", "8c8b41dab8264873dde739499bb8caf01ff632c65df98c5e38b5e0ca7dac8e88": "« é » n'est pas un caractère ASCII : relis le tableau d'UTF-8"}, "sign": 1}
📝 Ex B2.Q2a : réponse enregistrée (int).
WB_ANSWER {"hash": "3158f540764dd89dd181880744bdd33e47d777a3bf4556ae1fbf1ad7fdae8000", "id": "B2.Q2b", "kind": "int", "magnitude": 0, "mistakes": {"40ff222536af8327c2254d469de8321ede7bb2f6a6f6bccaf113c1fead75986b": "on compte ici des points de code, pas des octets", "9fe338556ad1cb2f47064cf61b1381fba594c6fa3578423edde489d5e754e018": "relis ce que fait NFD d'une lettre accentuée"}, "sign": 1}
📝 Ex B2.Q2b : réponse enregistrée (int).
WB_ANSWER {"hash": "089d0efa934344fc2ed148abd60d9208fbffcfc4acbfcf197002e5be5e17ceea", "id": "B2.Q2

**B2.Q3 — NFC ou NFD : deux chaînes identiques à l'écran, différentes en mémoire**

In [5]:
wb.record("B2.Q3a", False, mistakes={"une chaîne en NFC et sa forme NFD n'ont pas les mêmes points de code : que compare == ?": True})
wb.record("B2.Q3b", "C", mistakes={"casefold() change la casse, pas les accents": "A", "en NFC, « é » est un seul point de code au-delà de 127 : tu supprimerais la lettre entière": "B"})
wb.record("B2.Q3c", False, mistakes={"relis ce que NFKC fait de « ﬁ » et de « ² » : peut-on revenir en arrière ?": True})
wb.record("B2.Q3d", "A", mistakes={"BPE compare des points de code, pas ce qui s'affiche : relis le piège classique du §202.2": "B", "un tokenizer ne vérifie pas la normalisation : il découpe sans rien dire": "C"})
wb.record("B2.Q3e", True, mistakes={"NFC recompose tout ce que NFD a décomposé : une chaîne déjà composée revient-elle à l'identique ?": False})

WB_ANSWER {"hash": "9f6b2c748ff94d72c89d2a4ea55efbd9931c30d3217bbcb005ef3dbbfe25b258", "id": "B2.Q3a", "kind": "bool", "mistakes": {"c32e9ba74032272c0f8f9d7ecd9e70c5a06c74089f4138c76542aa0c1ca2b822": "une chaîne en NFC et sa forme NFD n'ont pas les mêmes points de code : que compare == ?"}}
📝 Ex B2.Q3a : réponse enregistrée (bool).
WB_ANSWER {"hash": "d124bca03870d543890a05d9f4c2e3b6e71269e72e5635d04e26783e2266322b", "id": "B2.Q3b", "kind": "str", "mistakes": {"81beb6258e133afa53aae221f3165471183e311e8283a2aa526d505f87ee1096": "en NFC, « é » est un seul point de code au-delà de 127 : tu supprimerais la lettre entière", "e87fb0d55f84e82f2eb4bdd96e0724f1cf9cbf75bc6932aa040fef4c3879d092": "casefold() change la casse, pas les accents"}}
📝 Ex B2.Q3b : réponse enregistrée (str).
WB_ANSWER {"hash": "0266554fbca9e2e057e06ac7a7509aded4aa016e2dacca6eda594a2a6d67f187", "id": "B2.Q3c", "kind": "bool", "mistakes": {"cc4ecf1d2c2d38e1e4c5864fc7be37274606c6ba47af85f0460054eac664eb58": "relis ce que NF

**B2.Q4 — Fusions BPE : pourquoi la liste ordonnée des fusions est le modèle**

In [6]:
wb.record("B2.Q4a", "B", mistakes={"deux tokenizers aux mêmes tokens peuvent découper autrement : relis « Encoder un mot nouveau »": "A", "le corpus sert à apprendre les fusions, pas à encoder": "C"})
wb.record("B2.Q4b", False, mistakes={"relis l'entraînement (§202.3) : une fusion franchit-elle la frontière d'un mot ?": True})
wb.record("B2.Q4c", 3 * 2, mistakes={"compte toutes les occurrences de la paire dans le mot": 3,
          "multiplie par la fréquence du mot": 2,
          "« n a » est une autre paire : on ne compte que « a » suivi de « n »": 9})
wb.record("B2.Q4d", "A", mistakes={"une fusion crée un seul token nouveau": "B", "les deux tokens fusionnés restent dans le vocabulaire : d'autres mots s'en servent": "C"})
wb.record("B2.Q4e", True, mistakes={"relis l'encadré 🕰️ sur le départage des fusions dans tokenizers": False})

WB_ANSWER {"hash": "db39c04a9c29b5e97678da5ec6f707985afc45ec58b51b3baef171666161bfd9", "id": "B2.Q4a", "kind": "str", "mistakes": {"c5cc3765c28781ca4795232d249cf1ce74ea55317915b7baa57281dfd92b12e3": "deux tokenizers aux mêmes tokens peuvent découper autrement : relis « Encoder un mot nouveau »", "d570b490974afb131450f2c7e802529433c5672cf04ee2fed942a97987041d75": "le corpus sert à apprendre les fusions, pas à encoder"}}
📝 Ex B2.Q4a : réponse enregistrée (str).
WB_ANSWER {"hash": "f64de4336ca9887ab5c5a239d6b697c8a7a9ec8d4313066d9c87bdf5640e139e", "id": "B2.Q4b", "kind": "bool", "mistakes": {"0fbcb5eea642ecdf3a19e5afecd53c056e5d6ed7ebae704db6512fa70ae27807": "relis l'entraînement (§202.3) : une fusion franchit-elle la frontière d'un mot ?"}}
📝 Ex B2.Q4b : réponse enregistrée (bool).
WB_ANSWER {"hash": "e3b1d424e87fef7d7c4d59114242ec71bd96d4690e4db8669fd3310097f1f250", "id": "B2.Q4c", "kind": "int", "magnitude": 0, "mistakes": {"46a72737076c608bfbe18136fe3966392c22dcc7464f2904131f6d95a7cb4

**B2.Q5 — BPE au niveau octet : jamais de token inconnu, vrai ou faux ?**

In [7]:
wb.record("B2.Q5a", True, mistakes={"au niveau octet, quel alphabet couvre tout texte UTF-8 ?": False})
wb.record("B2.Q5b", len('\u4e2d'.encode('utf-8')), mistakes={"sans fusion, chaque octet est un token : compte les octets du caractère": 1,
          "relis dans quelle plage du tableau d'UTF-8 tombent les caractères chinois": 4})
wb.record("B2.Q5c", False, mistakes={"relis l'encadré ⚠️ « un BPE qui jette des caractères » (§202.5)": True})
wb.record("B2.Q5d", "C", mistakes={"GPT-2 n'a que 50 257 tokens, et aucun inconnu": "A", "décoder la liste entière des identifiants redonne le texte exact": "B"})
wb.record("B2.Q5e", True, mistakes={"relis ce que donne le décodage d'un seul des deux octets de « Ç » (§202.4)": False})

WB_ANSWER {"hash": "352d563e8eeeb4d28f570e31db14867bb46b9067644b4e2bf6f5fa1859c4cc49", "id": "B2.Q5a", "kind": "bool", "mistakes": {"9f6d38ae4ddb0770d14fb900326c51f0633c5025c2fc58e3948c5dea678f373c": "au niveau octet, quel alphabet couvre tout texte UTF-8 ?"}}
📝 Ex B2.Q5a : réponse enregistrée (bool).
WB_ANSWER {"hash": "0e2273c60b7e041d867911c50cad33fd3c08686c21c2f3077b4993499a0c1549", "id": "B2.Q5b", "kind": "int", "magnitude": 0, "mistakes": {"341b24d882922c0809cfa0d171db5313480207e9303914566bc9cb2facd433fd": "sans fusion, chaque octet est un token : compte les octets du caractère", "a74a987e25384088d2e4655698de47dfa585881274d615f5972e343252466d67": "relis dans quelle plage du tableau d'UTF-8 tombent les caractères chinois"}, "sign": 1}
📝 Ex B2.Q5b : réponse enregistrée (int).
WB_ANSWER {"hash": "f9a0109413f7781c2b24fb7e9a651f1da9227dbc2ecccd5044e6a6b0d451f6fe", "id": "B2.Q5c", "kind": "bool", "mistakes": {"cbad562bfdc75c134786798338ced2bfce622744d86eba0b3e51687056fb855c": "relis l'

**B2.Q6 — Tokens spéciaux et espace initial : « Ġthe » n'est pas « the »**

In [8]:
wb.record("B2.Q6a", False, mistakes={"relis la table octet → caractère : l'espace fait-elle partie du token ?": True})
wb.record("B2.Q6b", "B", mistakes={"GPT-2 n'a qu'un token spécial, et il ne s'écrit pas ainsi": "A", "relis la table octet → caractère de GPT-2": "C"})
wb.record("B2.Q6c", "C", mistakes={"un token spécial n'est jamais découpé": "A", "le modèle a besoin de ce signal : on ne le supprime pas": "B"})
wb.record("B2.Q6d", True, mistakes={"relis le motif de GPT-2 : chaque caractère du texte appartient-il à un morceau ?": False})
wb.record("B2.Q6e", 1, mistakes={"relis la ligne EOS du tableau des tokens spéciaux": 0,
          "relis le paragraphe sous le tableau des tokens spéciaux": 2},
          fractional="on demande un nombre de tokens")

WB_ANSWER {"hash": "d381eb5c812d79a9bbe41cf9696aa64536146dcb270f0f19b0358f01069d4703", "id": "B2.Q6a", "kind": "bool", "mistakes": {"034d422dff21c2322ee72972543a0d021fa5ba67c41b47bfe6dafdb16ba03fbd": "relis la table octet → caractère : l'espace fait-elle partie du token ?"}}
📝 Ex B2.Q6a : réponse enregistrée (bool).
WB_ANSWER {"hash": "cb040d9219c5cd055ea0ea58f45e4e4eb579e373d83e51c6c00acd20e7e8720b", "id": "B2.Q6b", "kind": "str", "mistakes": {"2a96b91c33d94388c7eae880fcb81e2d6e4e12de9cba98a4fabec18e082561dd": "relis la table octet → caractère de GPT-2", "4bdfe784ece13962f4e0b73ae7f8abf6e655e205546a9e3958377e8264cd4b33": "GPT-2 n'a qu'un token spécial, et il ne s'écrit pas ainsi"}}
📝 Ex B2.Q6b : réponse enregistrée (str).
WB_ANSWER {"hash": "40b0d8575877b143fe65c8ec0ef87e8ad7e535c0fe6f4a8bafd81ed26a1153e4", "id": "B2.Q6c", "kind": "str", "mistakes": {"09d4993a48bf8606cbf68a80147a7c12ea5c5509da94df63310a9ca384902a94": "un token spécial n'est jamais découpé", "ea0cceffecf7e81f02a8caff85

**B2.Q7 — WordPiece, Unigram, SentencePiece : quel critère pour quelle méthode ?**

In [9]:
wb.record("B2.Q7a", "C", mistakes={"BPE part des caractères et ajoute des fusions": "A", "WordPiece part aussi des caractères et fusionne": "B"})
wb.record("B2.Q7b", "A", mistakes={"calcule le score de WordPiece des deux paires, pas leur compte": "B"})
wb.record("B2.Q7c", "B", mistakes={"BPE ne regarde que le compte de la paire": "A"})
wb.record("B2.Q7d", False, mistakes={"relis ce que SentencePiece fait des espaces (§202.5)": True})
wb.record("B2.Q7e", "A", mistakes={"les tokens spéciaux de BERT s'écrivent entre crochets": "B", "relis l'exemple de « unbelievably »": "C"})

WB_ANSWER {"hash": "1ef88f16c0796d413a95fbaf79c3c28b4c9f67b96a3887bb350ce08679fc1913", "id": "B2.Q7a", "kind": "str", "mistakes": {"3d67ced8f655b299b8aebfcfa78275058a22070c3f417ffe468df726f8d04bf3": "BPE part des caractères et ajoute des fusions", "d1de021b0f5c3ecd0ff97d4d8470ebea341f7585e4426b152b41c672f4832302": "WordPiece part aussi des caractères et fusionne"}}
📝 Ex B2.Q7a : réponse enregistrée (str).
WB_ANSWER {"hash": "bdd770b77f3ae0b75511992c5f87b2ba872d362b8817643605e04b903f1feedd", "id": "B2.Q7b", "kind": "str", "mistakes": {"e25b2fcda65d4e45ece7b8634c0c3d94b10062069643959def212595effeb7a9": "calcule le score de WordPiece des deux paires, pas leur compte"}}
📝 Ex B2.Q7b : réponse enregistrée (str).
WB_ANSWER {"hash": "860e9478a671c3a22ba9105b3605edb061f50ef9e4ff1d92b1d2347003e7e76c", "id": "B2.Q7c", "kind": "str", "mistakes": {"7d0b49e6763c1aa317130db2c4c4498a1e392fcd0042a71d46db8e091ab9df36": "BPE ne regarde que le compte de la paire"}}
📝 Ex B2.Q7c : réponse enregistrée (str).

**B2.Q8 — Pourquoi un LLM compte mal les lettres d'un mot et calcule mal avec les chiffres**

In [10]:
wb.record("B2.Q8a", "C", mistakes={"le modèle ne voit que des identifiants de tokens": "A", "même un BPE octet fusionne les octets des mots fréquents : relis « Les lettres »": "B"})
wb.record("B2.Q8b", False, mistakes={"compare les découpages de « 1234567 » et de « 1234656 » par GPT-2 (§202.6)": True})
wb.record("B2.Q8c", len('2026'), mistakes={"Qwen3 isole chaque chiffre": 1,
          "ce sont des groupes à la façon de Llama 3 : relis la règle de Qwen3": 2,
          "sans espace devant le nombre, pas de token d'espace": 5})
wb.record("B2.Q8d", math.ceil(len('12345678') / 3), mistakes={"le dernier groupe peut être plus court que les autres : il compte aussi": 2,
          "on compte des groupes, pas des chiffres": 8,
          "des groupes d'au plus 3 chiffres, pas de 2": 4})
wb.record("B2.Q8e", True, mistakes={"relis la fin du paragraphe « Les nombres » (§202.6)": False})

WB_ANSWER {"hash": "9078763cb06bfc83d8c0e5c3aaf615858a662e6ecd96c2e4450d9deb65928a00", "id": "B2.Q8a", "kind": "str", "mistakes": {"07f2364461e7d9633b22857edabf6a709e7e52c44d16abd77a96141705d20944": "même un BPE octet fusionne les octets des mots fréquents : relis « Les lettres »", "9bce975e19f4ee98c759880a515b4f1cf2e1c0bdbf8676cecafd227ae4dc2897": "le modèle ne voit que des identifiants de tokens"}}
📝 Ex B2.Q8a : réponse enregistrée (str).
WB_ANSWER {"hash": "acf359a6308581081e884a459dd9a0906899c40b8bd5493c43c86a16db647d58", "id": "B2.Q8b", "kind": "bool", "mistakes": {"b94175e25afd625f0b4e9797808e811962d6bdee5e6c645c401159c04032caff": "compare les découpages de « 1234567 » et de « 1234656 » par GPT-2 (§202.6)"}}
📝 Ex B2.Q8b : réponse enregistrée (bool).
WB_ANSWER {"hash": "f3f8fbe14f986137f089b2cc0f6a66e4af4c89e14d46de91a7d398b1f283f872", "id": "B2.Q8c", "kind": "int", "magnitude": 0, "mistakes": {"1ff8e934abeb650630684c4b54132d79ef24bcadfd27d7ec2b2934cece9b8cbe": "Qwen3 isole chaque

**B2.Q9 — nn.Embedding : une table de consultation entraînable, pas une couche magique**

In [11]:
wb.record("B2.Q9a", "(8, 20, 64)", mistakes={"n'oublie pas la dimension de l'embedding": "(8, 20)",
          "la table a 1 000 lignes, mais chaque id n'en lit qu'une": "(8, 20, 1000)",
          "les axes des ids restent dans le même ordre": "(20, 8, 64)"})
wb.record("B2.Q9b", 5000 * 32, mistakes={"une table de V lignes et de d colonnes : multiplie": 5032,
          "nn.Embedding n'a pas de biais": 160032})
wb.record("B2.Q9c", True, mistakes={"relis le calcul du one-hot fois la table (§202.7) : quelle ligne lit-il ?": False})
wb.record("B2.Q9d", "A", mistakes={"règle de la chaîne : les contributions des occurrences s'additionnent": "B", "chaque occurrence contribue au gradient de la ligne": "C"})
wb.record("B2.Q9e", True, mistakes={"relis le paragraphe sous l'exemple de nn.Embedding : une ligne jamais lue reçoit-elle un gradient ?": False})

WB_ANSWER {"hash": "a9e4b2674327bf25261bd67ca5ab7d0c3030e6cf5f3596becda4e2a7870c6e4e", "id": "B2.Q9a", "kind": "str", "mistakes": {"1b893e90b4591f7cbd49c90d3ff9fe5fbdf344a24d5d98c8c0cdb2d5d6f29d9c": "n'oublie pas la dimension de l'embedding", "32b0a5bebb6ffb1a292da976d3898c1c4f8756d4a33205aff5c9f11019d7c076": "la table a 1 000 lignes, mais chaque id n'en lit qu'une", "503388846d526d77f4c20c088ab70cdd1e95d5bb735bf9520dabf45c8638f01a": "les axes des ids restent dans le même ordre"}}
📝 Ex B2.Q9a : réponse enregistrée (str).
WB_ANSWER {"hash": "e8a4f2b8544c05c50fb0b90ed52601db6788129e52314974e405f1316eae930a", "id": "B2.Q9b", "kind": "int", "magnitude": 5, "mistakes": {"12f2e99573bedddc4ff703ffcb8d31529ebda51fda6aaac4a246466803dc400f": "nn.Embedding n'a pas de biais", "c1ee7826ca7cf62e0a98010ffa45fc73d84747aafc5ca8e4ba4016d8fcbafc7a": "une table de V lignes et de d colonnes : multiplie"}, "sign": 1}
📝 Ex B2.Q9b : réponse enregistrée (int).
WB_ANSWER {"hash": "686e6a7d0ceb66ca86ad86f5d2c5b9

**B2.Q10 — L'hypothèse distributionnelle : un mot se définit par ses voisins**

In [12]:
wb.record("B2.Q10a", "B", mistakes={"l'hypothèse distributionnelle ne regarde pas l'orthographe": "A", "se suivre n'est pas partager des contextes : « boire » et « café » se suivent, « thé » et « café » ont les mêmes voisins": "C"})
wb.record("B2.Q10b", True, mistakes={"relis le sens du signe de la PMI dans l'encadré 🧮 du §202.8": False})
wb.record("B2.Q10c", PMI_Q10, decimals=2, mistakes={"c'est le rapport des probabilités : la PMI est son logarithme": 1.0,
          "divise chaque compte par N avant de faire le rapport": math.log(6 / (30 * 200))})
wb.record("B2.Q10d", "C", mistakes={"prédire les contextes à partir du mot central, c'est le skip-gram": "A", "les mots tirés au hasard sont les négatifs de l'échantillonnage négatif": "B"})
wb.record("B2.Q10e", False, mistakes={"relis la règle du sous-échantillonnage : quels mots sont jetés le plus souvent ?": True})

WB_ANSWER {"hash": "35bbe50127ed5fddddb0ca8f68b45f401c5a478aca6196c04fc8c6cdf55a1a8d", "id": "B2.Q10a", "kind": "str", "mistakes": {"6f6648f9b7db168c2eaedb5fea1b7a0e3b47ffd1287aa00282bebe4d374cdbea": "se suivre n'est pas partager des contextes : « boire » et « café » se suivent, « thé » et « café » ont les mêmes voisins", "d7f9a77ac221a018095977944d9a7d64076960e99e7c8ae47d190aa96b5f9239": "l'hypothèse distributionnelle ne regarde pas l'orthographe"}}
📝 Ex B2.Q10a : réponse enregistrée (str).
WB_ANSWER {"hash": "82a9aca89395f4e953a609ee8b87198cdd714542fc03533a10b832cef8f0f4cb", "id": "B2.Q10b", "kind": "bool", "mistakes": {"6195730aee5e8dfdd343c0273f30995abe8fa7a972c62ff1ea727579ded21500": "relis le sens du signe de la PMI dans l'encadré 🧮 du §202.8"}}
📝 Ex B2.Q10b : réponse enregistrée (bool).
WB_ANSWER {"decimals": 2, "hash": "0fc00556874db3aed5f72ddb2cbe70c532f1228a03b1257a44ae86c09dab0a9e", "hash_coarse": "7396f8f3a752cfc31f5e45a97f84a0fccf61bcc7c0b4a9aecbfb002283c7cd58", "id": "B2.

**B2.Q11 — Statique ou contextuel : l'« avocat » du tribunal et celui du marché**

In [13]:
wb.record("B2.Q11a", "A", mistakes={"un embedding statique ne regarde pas la phrase": "B", "le vecteur d'un mot du vocabulaire n'est pas nul": "C"})
wb.record("B2.Q11b", "B", mistakes={"la ligne de la table est le point de départ ; la sortie du dernier bloc dépend de la phrase": "A", "un encodeur rend un vecteur par token": "C"})
wb.record("B2.Q11c", True, mistakes={"relis le paragraphe « Un mot, plusieurs sens » (§202.10)": False})
wb.record("B2.Q11d", "C", mistakes={"prendre un seul token, c'est un autre pooling : relis la formule du mean pooling": "A", "on moyenne les sorties du réseau, pas les lignes de la table": "B"})
wb.record("B2.Q11e", S_Q11[0], decimals=1, mistakes={"le masque retire le [PAD] de la moyenne : divise par le nombre de vrais tokens": 1.5,
          "on fait la moyenne des vecteurs, pas leur somme": 6.0})

WB_ANSWER {"hash": "971a6f6872b4344584be505ad60feb24b2ce409ccb8f4771b076ab1e081ca5b1", "id": "B2.Q11a", "kind": "str", "mistakes": {"17049f475de8ec98c2e53258fede56da872d7d5029bde66aa69bad70a74d0078": "le vecteur d'un mot du vocabulaire n'est pas nul", "7aec80aa04c382069dfd119373091a02199705324c2a4986d2e9fc7c8a48daf5": "un embedding statique ne regarde pas la phrase"}}
📝 Ex B2.Q11a : réponse enregistrée (str).
WB_ANSWER {"hash": "e592ad4391a9bb6796e860446b662a48e00449dd62c2d54b052cf3336828bc39", "id": "B2.Q11b", "kind": "str", "mistakes": {"57d7add3136887829b9b0d8c662abf2475d4ecbce7ed1293874bad7649eb93f1": "la ligne de la table est le point de départ ; la sortie du dernier bloc dépend de la phrase", "ded8cfd41d56d07db4b082af62602cad89d53faace52967ecc1957f4bf52d13e": "un encodeur rend un vecteur par token"}}
📝 Ex B2.Q11b : réponse enregistrée (str).
WB_ANSWER {"hash": "477c22b8398e8b1283cf6753c3b181b673210a1c825dd45bdec5a1a6cb73902f", "id": "B2.Q11c", "kind": "bool", "mistakes": {"94e7d1

**B2.Q12 — Cosinus ou distance euclidienne pour des embeddings normalisés ?**

In [14]:
wb.record("B2.Q12a", 2 - 2 * 0.8, decimals=1, mistakes={"on demande la distance au carré, pas la distance": math.sqrt(0.4),
          "relis la relation de la fiche : un facteur 2 manque": 1 - 0.8,
          "le cosinus est multiplié par 2 dans la relation": 2 - 0.8})
wb.record("B2.Q12b", True, mistakes={"développe ‖a − b‖² pour deux vecteurs de norme 1": False})
wb.record("B2.Q12c", math.sqrt(2 - 2 * 0), decimals=3, mistakes={"on demande la distance, pas son carré": 2.0,
          "deux vecteurs orthogonaux ne sont pas confondus : applique la relation avec leur cosinus": 0.0})
wb.record("B2.Q12d", False, mistakes={"un vecteur de grande norme gagne au produit scalaire sans changer de direction": True})
wb.record("B2.Q12e", -1.0, decimals=1, mistakes={"(−6, −8) va dans le sens opposé : le signe compte": 1.0,
          "le cosinus divise le produit scalaire par les deux normes": -50.0})

WB_ANSWER {"decimals": 1, "hash": "08db6819ca540d403e2d42f349d0d8f6076b88437f6eadb4e0343c91896eb3d4", "hash_coarse": "093e5c58013e4387225afc4b02d26e03034d456d3e4c644fb7e883ffebe13332", "id": "B2.Q12a", "kind": "float", "magnitude": -1, "mistakes": {"095ba918fdf450cca33b1a80600b3abc1da934b39c0926d2915943cd2107abc7": "on demande la distance au carré, pas la distance", "3b6affb4c57b2bfaf3ff507f61a98fc09fea628f6e8d9b91f56a906360f282d7": "relis la relation de la fiche : un facteur 2 manque", "62211705969d255b13c7a4059b1cbfe0ba1de40f677a8d67e6e5393e24bdba5f": "le cosinus est multiplié par 2 dans la relation"}, "sign": 1}
📝 Ex B2.Q12a : réponse enregistrée (float, 1 décimale(s)).
WB_ANSWER {"hash": "379b8b732127ca7360f1d2e0124d268335216848b5f654c3ac4e06c84fc944bd", "id": "B2.Q12b", "kind": "bool", "mistakes": {"5cdaf29ced510a0f75561cacd6a5767eda230d2116a4143101ea4acfa6763870": "développe ‖a − b‖² pour deux vecteurs de norme 1"}}
📝 Ex B2.Q12b : réponse enregistrée (bool).
WB_ANSWER {"decimals"

**B2.R1 — À quoi sert l'embedding d'image extrait d'un réseau pré-entraîné ?**

In [15]:
wb.record("B2.R1a", True, mistakes={"relis le §201.3 de B1 : combien de fois calcule-t-on un embedding figé ?": False})
wb.record("B2.R1b", 512, mistakes={"c'est le nombre de sorties de la tête ImageNet : on demande l'entrée de fc": 1000,
          "c'est la dimension de l'embedding de ResNet-50": 2048})
wb.record("B2.R1c", "A", mistakes={"un fine-tuning modifie le backbone : l'embedding n'est plus figé": "B", "un k plus proches voisins n'apprend aucun poids": "C"})
wb.record("B2.R1d", False, mistakes={"relis pourquoi B1 compare les embeddings par le cosinus (§201.3)": True})
wb.record("B2.R1e", "B", mistakes={"des pixels bruts ne sont pas un embedding": "A", "la tête produit des scores de classes, pas une représentation": "C"})

WB_ANSWER {"hash": "4484d2620f7a834ac9e439534569499f2e3086e2f148f6c3f94b676410eb974a", "id": "B2.R1a", "kind": "bool", "mistakes": {"43c84fb878065dcc29d64fb03d328877f0b87b0252e583cdbba0e081451cfe12": "relis le §201.3 de B1 : combien de fois calcule-t-on un embedding figé ?"}}
📝 Ex B2.R1a : réponse enregistrée (bool).
WB_ANSWER {"hash": "af2c9a9a953b3a7ef0d521ddc1626e83953c1bbe6ec563c465242fee17f6f96e", "id": "B2.R1b", "kind": "int", "magnitude": 2, "mistakes": {"2fbc35c007ea194bf0905d4ebe539d638b1fff97980217bd3ae812fd01367e2c": "c'est la dimension de l'embedding de ResNet-50", "4c3ed326dfda516348ca5ca52ad5e35fb28acaab1afba3f43125a309bbd26cec": "c'est le nombre de sorties de la tête ImageNet : on demande l'entrée de fc"}, "sign": 1}
📝 Ex B2.R1b : réponse enregistrée (int).
WB_ANSWER {"hash": "ef5361ebda0f49ec38d675390f586e7148b5f10133e124bbc0f2adbaf00fdde8", "id": "B2.R1c", "kind": "str", "mistakes": {"1d1b809cc97bf35ed2446ac1e3dcfc89dc9d0aa0c6dff67747935336f2c9be80": "un k plus proches

**B2.R2 — build_vocab, encode, decode et le token inconnu**

In [16]:
wb.record("B2.R2a", 2, mistakes={"les ids suivent la fréquence décroissante : quelle lettre est la plus fréquente ?": 0,
          "les ids commencent à 0": 3})
wb.record("B2.R2b", [3, 1, 2, 0, 1, 2, 1], mistakes={"avec unk_token, le token inconnu prend l'id 0 et décale tous les autres": [2, 0, 1, 0, 0, 1, 0],
          "un caractère absent du vocabulaire ne reçoit pas de nouvel id : il devient le token inconnu": [3, 1, 2, 4, 1, 2, 1]})
wb.record("B2.R2c", False, mistakes={"que devient « d » à l'aller, et que rend decode pour l'id du token inconnu ?": True})
wb.record("B2.R2d", "C", mistakes={"relis le rôle de unk_token dans encode": "A", "build_vocab ne découpe jamais un mot : c'est le travail d'un BPE": "B"})
wb.record("B2.R2e", 0, mistakes={"max_size compte aussi le token inconnu : « chat » est-il resté dans le vocabulaire ?": 2,
          "relis l'ordre des ids : le token inconnu vient d'abord, puis le mot le plus fréquent": 1})

WB_ANSWER {"hash": "3804cb8a8ed4f1f9df75df116d8eacb95967e336798038a744d205bcb1cc1d8c", "id": "B2.R2a", "kind": "int", "magnitude": 0, "mistakes": {"6ee190147c852dffca2dc203f6694cb526c2f424dc7a877c8e94926d41c04af8": "les ids commencent à 0", "acc44e4e3b558e9a8325815d4a94f36f213009c4cb5cfd5dca9f628f8af4e450": "les ids suivent la fréquence décroissante : quelle lettre est la plus fréquente ?"}, "sign": 1}
📝 Ex B2.R2a : réponse enregistrée (int).
WB_ANSWER {"decimals": 0, "element_hashes": ["cfbe645c12642d4cb83d77bdca04ae53f8330750a91766f49ce19d8d93b4db4e", "d3bb14836b8e67d507aaea6f9c94fd8f1954104f63d751b30a47d7ae8cc3dafa", "e7510a873f9010dd640323a1f245cec3b7129ff9afc3497873c6545d73c869df", "62b4939dad7b349012e094cf3c5a04d2d4affc10d16443defda5efbd46726b49", "e39b098a1db62a60e720f88910a6601dd355a7b093b7c71f6d6289a6e1130301", "84ec0ca5882968e0fc064b6ca721fc1befea6ee8ddf9a0ac5effd4f530b07a87", "18669220f9d987abd8c98eb969f79b1f1f29d0cdf4e78fdc4498c575b3405a33"], "hash": "29a8497209797952ecaa6e

**B2.R3 — Entropie, Huffman et ce que l'on compresse vraiment**

In [17]:
wb.record("B2.R3a", H_R3, decimals=2, mistakes={"l'entropie se mesure ici en bits : logarithme de base 2": -(P_R3 * np.log(P_R3)).sum(),
          "n'oublie pas le signe moins de la formule": -H_R3})
wb.record("B2.R3b", 1, mistakes={"le symbole le plus probable reçoit le mot de code le plus court": 3,
          "un code de Huffman n'est pas de longueur fixe": 2})
wb.record("B2.R3c", True, mistakes={"relis le lien avec le ch. 6 au début du §202.3": False})
wb.record("B2.R3d", False, mistakes={"relis la borne du ch. 6 : l'entropie est-elle un plancher ou un plafond ?": True})
wb.record("B2.R3e", 4000 / (1000 * 8), decimals=1, mistakes={"le taux du ch. 6 met les bits envoyés au numérateur": 2.0,
          "un octet vaut 8 bits : compare des bits à des bits": 4.0})

WB_ANSWER {"decimals": 2, "hash": "e9328103b3c68969312fe96cae486b7c1d7446b3c5583b8c39ef892a5c085ea5", "hash_coarse": "9127d4b134519209b3386a694d299135b0d1149d88000bafc345eb78d8528722", "id": "B2.R3a", "kind": "float", "magnitude": 0, "mistakes": {"86ae6474a389e20c3884c600a81fece1e51e032ead986cfe67cfedc28736e509": "l'entropie se mesure ici en bits : logarithme de base 2", "e5dc06cefa2f3bbfd4498a2358cd4ee48fb3367147f78f8079f3ebf1ce44a502": "n'oublie pas le signe moins de la formule"}, "sign": 1}
📝 Ex B2.R3a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"hash": "f12730af284331acb6fc1edd68202effffba570c9147666551bef92d418e0708", "id": "B2.R3b", "kind": "int", "magnitude": 0, "mistakes": {"79b2ab3292ed6144c8bec6fa072a4de6005f2a0aa85c03938fd6e0cfa594a813": "le symbole le plus probable reçoit le mot de code le plus court", "a30a31157136a795ec579289bedc6b0de531e70f33a0984dc30636f9e8db6d25": "un code de Huffman n'est pas de longueur fixe"}, "sign": 1}
📝 Ex B2.R3b : réponse enregistré

**Ex B2.1 — « Noël à Paris » en UTF-8 : octets, points de code, longueur**

In [18]:
wb.record("B2.1a", len(S_B21), mistakes={"les espaces sont des points de code, eux aussi": 10,
          "relis la question : on compte des points de code, et en NFC une lettre accentuée n'en est qu'un": 14,
          "on compte des points de code, pas des mots": 3})
wb.record("B2.1b", len(B_B21), mistakes={"relis la question : on demande des octets, et une lettre accentuée en prend plus d'un": 12,
          "relis la question : on demande le texte tel qu'il est écrit, en NFC": 16,
          "« ë » et « à » sont sous U+0800 : relis les plages du tableau d'UTF-8": 18})
wb.record("B2.1c", list(B_B21[2:4]), mistakes={"c'est le point de code U+00EB écrit sur deux octets, pas son écriture UTF-8": [0x00, 0xEB],
          "le second octet commence par les bits 10": [0xC3, 0x2B],
          "le premier octet commence par les bits 110": [0x03, 0xAB],
          "l'octet qui porte les bits de poids fort vient en premier": [0xAB, 0xC3]})
wb.record("B2.1d", B_B21[3], mistakes={"les indices d'un bytes comptent les octets, pas les caractères : « ë » occupe deux positions": ord("l"),
          "les octets sont numérotés à partir de 0": B_B21[2],
          "c'est le point de code de « ë » : on demande un octet de son écriture UTF-8": 0xEB})
wb.record("B2.1e", len(NFD_B21.encode('utf-8')), mistakes={"relis la question : on demande les octets du texte décomposé (NFD)": 14,
          "U+0300 et U+0308 sont sous U+0800 : deux octets chacun": 18,
          "en NFD, chaque lettre accentuée devient deux points de code : compte les octets des deux": 12})
wb.record("B2.1f", utf8_two_bytes(0x153), mistakes={"c'est le point de code U+0153 écrit sur deux octets, pas son écriture UTF-8": [0x01, 0x53],
          "le premier octet porte les 5 bits de poids fort du point de code : il ne vaut pas toujours C3": [0xC3, 0x93],
          "le second octet commence par les bits 10": [0xC5, 0x13],
          "le premier octet commence par les bits 110": [0x05, 0x93]})
wb.record("B2.1g", False, mistakes={"sans errors=, que fait decode d'octets qui ne forment pas de l'UTF-8 valide ?": True})

WB_ANSWER {"hash": "b71525de4b98bc5d1e430b29fe88c001c4585f1d65594be750ec1b8a1316b725", "id": "B2.1a", "kind": "int", "magnitude": 1, "mistakes": {"153a2cc7db45dca39297f8d47baf2ec8c00109362dc1f7bf83f42b5fff6be871": "relis la question : on compte des points de code, et en NFC une lettre accentuée n'en est qu'un", "187810e80073dcba293c7ca3a5224fdb1fe33ea1b208ba2be0035c31f87194a9": "on compte des points de code, pas des mots", "b41c45d1847c1e4946e7e826fb9004ec27c961e6137ab9f55f389fb58ad607d1": "les espaces sont des points de code, eux aussi"}, "sign": 1}
📝 Ex B2.1a : réponse enregistrée (int).
WB_ANSWER {"hash": "dcd896f6cb77e56d5bffa805777869f2b2935700db2205ac50da83551a849322", "id": "B2.1b", "kind": "int", "magnitude": 1, "mistakes": {"304071f005487a7325d1c6f2e5c4e4d03b3f2d22cdece8430e63e95d5c189518": "« ë » et « à » sont sous U+0800 : relis les plages du tableau d'UTF-8", "3c99fd46c7b76f59b2ed613e3b0cde239a69fa56131895f696dd6e39b0d78d5d": "relis la question : on demande des octets, et u

**Ex B2.2 — Analogie à la main : roi − homme + femme sur des vecteurs de dimension 3**

In [19]:
wb.record("B2.2a", float(Q_B22 @ E_B22[1]), decimals=2, mistakes={"calcule d'abord la requête roi − homme + femme, puis son produit scalaire avec reine": float(E_B22[0] @ E_B22[1]),
          "la requête est roi − homme + femme : n'oublie pas de retrancher homme": float((E_B22[0] + E_B22[3]) @ E_B22[1])})
wb.record("B2.2b", COS_B22[1], decimals=3, mistakes={"divise aussi par la norme de la requête": float(Q_B22 @ E_B22[1]) / np.linalg.norm(E_B22[1]),
          "divise aussi par la norme de reine": float(Q_B22 @ E_B22[1]) / np.linalg.norm(Q_B22),
          "le produit scalaire seul n'est pas le cosinus : divise par les deux normes": float(Q_B22 @ E_B22[1])})
wb.record("B2.2c", COS_B22[3], decimals=3, mistakes={"divise aussi par la norme de la requête": float(Q_B22 @ E_B22[3]) / np.linalg.norm(E_B22[3]),
          "divise aussi par la norme de femme": float(Q_B22 @ E_B22[3]) / np.linalg.norm(Q_B22),
          "le produit scalaire seul n'est pas le cosinus : divise par les deux normes": float(Q_B22 @ E_B22[3])})
wb.record("B2.2d", analogy_b22(E_B22, "homme", "roi", "femme", exclude=False), mistakes={"relis la question : ici, aucun mot n'est exclu": "reine",
          "compare les cosinus des sept mots, à 3 décimales": "princesse",
          "relis la question : on compare des cosinus, qui divisent par les normes": "trône"})
wb.record("B2.2e", analogy_b22(E_B22, "homme", "roi", "femme"), mistakes={"on exclut les trois mots de la question, femme comprise": "femme",
          "compare les cosinus de reine et de princesse, à 3 décimales": "princesse",
          "relis la question : on compare des cosinus, qui divisent par les normes": "trône"})
wb.record("B2.2f", analogy_b22(E_B22, "homme", "roi", "femme", cosine=False), mistakes={"relis la question : on compare ici des produits scalaires bruts, sans diviser par les normes": "reine",
          "n'oublie pas l'exclusion des trois mots de la question": "femme"})
wb.record("B2.2g", analogy_b22(E_UNIT_B22, "homme", "roi", "femme") == "reine", mistakes={"recalcule les cosinus avec les vecteurs ramenés à la norme 1 : le premier mot non exclu change-t-il ?": False})

WB_ANSWER {"decimals": 2, "hash": "e2d21e745aa3c72e21211347b2c5129f0382e0a81908ac61fbebc568a2119f96", "hash_coarse": "f6bf8e6eba9735fb426cd6663bdff87f88b0b6b865610d743d09edf600311592", "id": "B2.2a", "kind": "float", "magnitude": 0, "mistakes": {"3e002340748d6ee234d68d2f55cd2be2408a2174757f2c6b4978e33794dbda2e": "calcule d'abord la requête roi − homme + femme, puis son produit scalaire avec reine", "833f36d64103e803c3b7429decfd99281bd6f025bc6bb38610255cdbc8906bc8": "la requête est roi − homme + femme : n'oublie pas de retrancher homme"}, "sign": 1}
📝 Ex B2.2a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "307e544233b905ee05e745c52d73f4c76d315bcd68395dcf92aaacaeb3264ea8", "hash_coarse": "bf6a8c92bcf42ece3ba1cf4125413ac6232cadc7653f810a774f3c42b38fc774", "id": "B2.2b", "kind": "float", "magnitude": -1, "mistakes": {"c3c18951c79193b338edd2cc7c33d18054f416cce28ef90e89668171112f9d8c": "divise aussi par la norme de reine", "cd1e8413fbf15eab3f2b4c96d4a0069441

**Ex B2.3 — Trois fusions BPE à la main sur un mini-corpus**

In [20]:
wb.record("B2.3a", len(ALPHABET_B23), mistakes={"« é » est un caractère de l'alphabet comme les autres": 4})
wb.record("B2.3b", ALPHABET_B23.index('é'), mistakes={"l'alphabet est trié par point de code, pas dans l'ordre alphabétique du français : où vient U+00E9 ?": 0,
          "les identifiants commencent à 0": 5})
wb.record("B2.3c", list(MERGES_B23[0]), mistakes={"à égalité, la règle prend la plus petite paire d'identifiants, pas la première lue dans le mot": [4, 3],
          "relis les identifiants de l'alphabet : il est trié par point de code": [0, 4],
          "les identifiants viennent de l'alphabet trié par point de code, et la paire se lit dans l'ordre du mot": [4, 0],
          "recompte cette paire : est-elle la plus fréquente ?": [4, 2]})
wb.record("B2.3d", COUNTS_B23[0], mistakes={"multiplie par la fréquence de chaque mot": 2,
          "on compte une seule paire, pas les deux paires à égalité": 14})
wb.record("B2.3e", list(MERGES_B23[1]), mistakes={"relis le départage de la première fusion": [5, 4],
          "après la première fusion, « t » et « é » ne sont plus voisins dans « été »": [4, 3],
          "recompte cette paire après la première fusion : est-elle la plus fréquente ?": [5, 2]})
wb.record("B2.3f", list(MERGES_B23[2]), mistakes={"à égalité, la plus petite paire d'identifiants passe d'abord : quel est l'identifiant du token fusionné ?": [6, 2],
          "à égalité, compare d'abord les premiers identifiants des paires": [1, 4]})
wb.record("B2.3g", TOKENS_AFTER_B23, mistakes={"c'est le nombre de mots du corpus : compte les tokens de chaque mot": 10,
          "multiplie le nombre de tokens de chaque mot par sa fréquence": 8,
          "c'est le nombre de caractères : applique les trois fusions": 34})
wb.record("B2.3h", True, mistakes={"relis la règle d'arrêt : on s'arrête quand la meilleure paire apparaît moins de min_frequency fois": False})

WB_ANSWER {"hash": "a55af7c8f0e820f2884e7d20466e4fa79d9b98de42732fffd1af9048fb0dc272", "id": "B2.3a", "kind": "int", "magnitude": 0, "mistakes": {"08c13dc84df687aeb7171ffe486897955f1d20dffdecdb54abc8559f3b956776": "« é » est un caractère de l'alphabet comme les autres"}, "sign": 1}
📝 Ex B2.3a : réponse enregistrée (int).
WB_ANSWER {"hash": "efa74a39d503962acb00e7d6f4b9f8c2f1ec505e2dbeddd44e107b012fb7925a", "id": "B2.3b", "kind": "int", "magnitude": 0, "mistakes": {"c1122f0ab7f204b5ad5b381bb6939267924867fb46b0d058a34740fc2baf43ca": "les identifiants commencent à 0", "c3f1ce61f2c4c499b12fb331053dbd8c87dd55d2f378b9fcdd688c6af8a67fc4": "l'alphabet est trié par point de code, pas dans l'ordre alphabétique du français : où vient U+00E9 ?"}, "sign": 1}
📝 Ex B2.3b : réponse enregistrée (int).
WB_ANSWER {"decimals": 0, "element_hashes": ["01498542bf59bbe2f296ed6f287cd5e9684530e5a60a8a2806bd60ec58251426", "128ed23636c22200704d8519d420e89736450806d10ccdcc450595e245cf3115"], "hash": "d84af2136d3e1

**Ex B2.4 — Encoder un mot jamais vu avec des fusions déjà apprises**

In [21]:
wb.record("B2.4a", encode_b24("chantons"), mistakes={"applique les fusions jusqu'au bout : aucune paire voisine ne doit rester dans la table": [13, 12, 7]})
wb.record("B2.4b", encode_b24("ont"), mistakes={"relis la question : on applique les fusions par rang": encode_b24("ont", by_rank=False)})
wb.record("B2.4c", encode_b24("ont", by_rank=False), mistakes={"relis la question : on suit ici l'erreur de lecture, la paire connue la plus à gauche": encode_b24("ont")})
wb.record("B2.4d", encode_b24("chanteront"), mistakes={"après la fusion « nt », « o » et « n » sont-ils encore voisins ?": encode_b24("chanteront", by_rank=False),
          "applique toutes les fusions de la table, jusqu'au rang 7": [13, 14, 5, 9]})
wb.record("B2.4e", encode_b24("chantez"), mistakes={"relis où les tokens spéciaux se placent dans la numérotation (§202.3)": [13, 2, 0],
          "un caractère inconnu devient un token, il ne disparaît pas": [13, 2]})
wb.record("B2.4f", len("été".encode("utf-8")), mistakes={"au niveau octet, « é » s'écrit avec deux octets": 3,
          "« t » est un caractère ASCII : un seul octet": 6})
wb.record("B2.4g", False, mistakes={"que rend decode pour l'identifiant de <unk> ?": True})

WB_ANSWER {"decimals": 0, "element_hashes": ["e073d02906bbc5c131c48637a0aa2fff9fcb4324e0e191dabfc9a19733f64c4e", "f243a819de227e3154c0e17a09ff58e65f560c31614bbbf139a55866bbfc7cae"], "hash": "ef9dbf0a419fe4244a9dc8098dfaf48e7713b8745fefd0993af4f22874ef505b", "id": "B2.4a", "integer": true, "kind": "array", "mistakes": {"39aa4753e82ed6f215ab95a563d731891398cfdee17e1ccd20b3ad6c8bb39253": "applique les fusions jusqu'au bout : aucune paire voisine ne doit rester dans la table"}, "shape": [2]}
📝 Ex B2.4a : réponse enregistrée (array, 0 décimale(s)).
WB_ANSWER {"decimals": 0, "element_hashes": ["c5c72dc946e7b72820fe3a3c9984af7b885000e78e46b088e6f4bc30468005a8", "08d8242809e7e6f536763d3f196c2da3ef02811ce7a374f39ccfcbcedf688e7f"], "hash": "76e51d884459f055436b641c661206f4d3a962ff0f06bdd79e00a63218fc9fb9", "id": "B2.4b", "integer": true, "kind": "array", "mistakes": {"aa91c76a0feaef7738fb3de82e7d845721c8eb2a7ad606635b2145cc3e111510": "relis la question : on applique les fusions par rang"}, "shap

**Ex B2.5 — One-hot fois matrice : nn.Embedding à la main, et quelles lignes reçoivent un gradient**

In [22]:
wb.record("B2.5a", "(4, 5)", mistakes={"une ligne par position, une colonne par token du vocabulaire": "(5, 4)",
          "le one-hot a V colonnes, pas d": "(4, 3)"})
wb.record("B2.5b", OUT_B25[1].tolist(), mistakes={"les positions sont numérotées à partir de 0 : t = 1 est la deuxième": OUT_B25[0].tolist(),
          "le token lu à la position t = 1 est 0 : relis sa ligne de la table": E_B25[1].tolist()})
wb.record("B2.5c", L_B25, decimals=1, mistakes={"le token 3 apparaît deux fois : compte ses deux positions": -7.0})
wb.record("B2.5d", GRAD_B25[3].tolist(), mistakes={"le token 3 apparaît deux fois : sa ligne reçoit la somme des deux gradients": W_B25.tolist(),
          "le gradient de w · e par rapport à e est w, pas e": (2 * E_B25[3]).tolist()})
wb.record("B2.5e", int((np.abs(GRAD_B25).sum(axis=1) > 0).sum()), mistakes={"une ligne lue deux fois ne compte qu'une fois": 4,
          "les lignes jamais lues ne reçoivent rien": 5})
wb.record("B2.5f", (E_B25[3] - LR_B25 * GRAD_B25[3]).tolist(), decimals=2, mistakes={"un pas de SGD retranche η fois le gradient": (E_B25[3] + LR_B25 * GRAD_B25[3]).tolist(),
          "la ligne 3 reçoit la somme des gradients de ses deux positions": (E_B25[3] - LR_B25 * W_B25).tolist()})
wb.record("B2.5g", False, mistakes={"relis ce que fait padding_idx (§202.7)": True})
wb.record("B2.5h", True, mistakes={"relis ce que la fiche dit des lignes qui ne sont pas lues, et de l'optimiseur (§202.7)": False})

WB_ANSWER {"hash": "ae0ec679328119510feb0a2400c7afa275235d2ac4d78f24e8560a0af87ae2d1", "id": "B2.5a", "kind": "str", "mistakes": {"17fe530506e7eb0229f207f2713f3808accf587dfd5dd8c2c523b65fbc2fee13": "une ligne par position, une colonne par token du vocabulaire", "48b4871257af557b6ebba767ccacb65969af5df001b80bc288867cfb73f0943d": "le one-hot a V colonnes, pas d"}}
📝 Ex B2.5a : réponse enregistrée (str).
WB_ANSWER {"decimals": 0, "element_hashes": ["1b8a9ae27e43a3866fa82566e982b5d199c3ca67561584b265a580611cd256ee", "726df3bf38732ba3e4dbbd2ad3f48053d3100faeda715b1ba0ca0c8debc60d8b", "49c4bf777fc2f158ae5ba83a0e960fa78038c88505f26e7647b0939faa7b25a7"], "hash": "dd50fc7a740b9e96f142850e3a0dd26cbb345aac484e8e547c0ef7faf4203be2", "id": "B2.5b", "integer": true, "kind": "array", "mistakes": {"58b6830e41f7b0734b99c49649d24dd1127c1d2bbfd119f2be4c29f33103fb5a": "le token lu à la position t = 1 est 0 : relis sa ligne de la table", "6106a694be33566fd0f1a29569760f79ae547d91022c7f1331241aaa46f9cb90": "

**Ex B2.6 — Co-occurrences et PPMI à la main (fenêtre de 1, quatre mots)**

In [23]:
wb.record("B2.6a", int(C_B26[0, 2]), mistakes={"la fenêtre est symétrique : compte aussi « le » à gauche de « chat »": 1,
          "la fenêtre est symétrique : compte aussi « le » à droite de « chat »": 2})
wb.record("B2.6b", int(N_B26), mistakes={"chaque paire voisine compte dans les deux sens": 7,
          "un texte de n tokens a n − 1 paires voisines": 16})
wb.record("B2.6c", pmi_b26("chat", "le"), decimals=3, mistakes={"la PMI est le logarithme du rapport": 2.1,
          "prends le logarithme naturel, pas en base 2": math.log2(2.1),
          "divise chaque somme par N, comme le compte de la paire": math.log(3 / 20)})
wb.record("B2.6d", pmi_b26("court", "le"), decimals=3, mistakes={"relis la question : on demande la PMI, avant le max à 0": 0.0,
          "la PMI est le logarithme du rapport": 14 / 15,
          "prends le logarithme naturel, pas en base 2": math.log2(14 / 15)})
wb.record("B2.6e", P_ALPHA_B26[WORDS_B26.index('le')], decimals=3, mistakes={"c'est p(le) sans lissage : élève les sommes des colonnes à la puissance α": 5 / 14})
wb.record("B2.6f", pmi_b26("chat", "le", alpha=0.75), decimals=3, mistakes={"relis la question : on demande la PPMI lissée, avec α = 0,75": pmi_b26("chat", "le"),
          "α ne porte que sur la loi des contextes, pas sur celle des mots": pmi_b26("chat", "le", 0.75, alpha_rows=True),
          "le mot est la ligne, le contexte la colonne : relis l'ordre": pmi_b26("le", "chat", 0.75)})
wb.record("B2.6g", False, mistakes={"compare PPMI(chat, le) et PPMI(le, chat) avec α = 0,75": True})
wb.record("B2.6h", True, mistakes={"recalcule PMI(court, le) avec p_α(le) au lieu de p(le)": False})

WB_ANSWER {"hash": "db69286d873e819295aa5ad42de762fc65af627767a2b9a15b017871d9ec8cc1", "id": "B2.6a", "kind": "int", "magnitude": 0, "mistakes": {"aa8429e62ff74e03f1e3508a1bc8b677531f90e25c8da773b5c224e52d22eaf7": "la fenêtre est symétrique : compte aussi « le » à droite de « chat »", "cbff766cf6a786c5617a9f005f0454b2c7ffdc9ad0856886334aa18fce5e6cca": "la fenêtre est symétrique : compte aussi « le » à gauche de « chat »"}, "sign": 1}
📝 Ex B2.6a : réponse enregistrée (int).
WB_ANSWER {"hash": "747a4471ce1bb29c5caa0ec0e4af19c94a671afe3838d1018769fc4615ada20f", "id": "B2.6b", "kind": "int", "magnitude": 1, "mistakes": {"0cd422500b2ae5847c64dc85054095461623d6fc5151d4ceaec56d3ecd5b6978": "chaque paire voisine compte dans les deux sens", "60188aa5ef66052de12bb889fdd0ce0f312ec7331ceb56afde19e0e7c0940ec9": "un texte de n tokens a n − 1 paires voisines"}, "sign": 1}
📝 Ex B2.6b : réponse enregistrée (int).
WB_ANSWER {"decimals": 3, "hash": "0eba98232fcff3bf1630a353ff7a896744ada8b7c0174cc59c2f80a

**Ex B2.9 — Fermi : poids de la table d'embedding de GPT-2 et d'un LLM à 128 000 tokens**

In [24]:
wb.record("B2.9a", GPT2_TABLE_B29, mistakes={"une table de V lignes et de d colonnes : multiplie": 50257 + 768,
          "GPT-2 lie ses poids : la tête de sortie est la même matrice que la table": 2 * GPT2_TABLE_B29})
wb.record("B2.9b", 100 * GPT2_TABLE_B29 / GPT2_TOTAL_B29, decimals=1, mistakes={"on demande un pourcentage, pas une proportion": GPT2_TABLE_B29 / GPT2_TOTAL_B29,
          "GPT-2 lie ses poids : la table n'est comptée qu'une fois": 200 * GPT2_TABLE_B29 / GPT2_TOTAL_B29})
wb.record("B2.9c", GPT2_TABLE_B29 * 4 / 1e6, decimals=1, mistakes={"un float32 occupe 4 octets": GPT2_TABLE_B29 / 1e6,
          "1 Mo vaut ici 10⁶ octets, pas 2²⁰": GPT2_TABLE_B29 * 4 / 2 ** 20,
          "un float32, c'est 32 bits, soit 4 octets : compte en octets": GPT2_TABLE_B29 * 32 / 1e6})
wb.record("B2.9d", L31_TABLE_B29, mistakes={"une table de V lignes et de d colonnes : multiplie": 128256 + 4096,
          "relis la question : on demande la table d'entrée seule": 2 * L31_TABLE_B29})
wb.record("B2.9e", 200 * L31_TABLE_B29 / L31_TOTAL_B29, decimals=1, mistakes={"compte les deux tables : Llama 3.1 8B ne lie pas ses poids": 100 * L31_TABLE_B29 / L31_TOTAL_B29,
          "on demande un pourcentage, pas une proportion": 2 * L31_TABLE_B29 / L31_TOTAL_B29})
wb.record("B2.9f", round(100 * L32_TABLE_B29 / L32_TOTAL_B29), mistakes={"Llama 3.2 1B lie ses poids : une seule table": round(200 * L32_TABLE_B29 / L32_TOTAL_B29),
          "on demande un pourcentage, pas une proportion": 0},
          fractional="arrondis le pourcentage à l'entier le plus proche")
wb.record("B2.9g", L31_TABLE_B29 * 2 > 1e9, mistakes={"un bfloat16 occupe 2 octets : multiplie le nombre de paramètres par 2": False})

WB_ANSWER {"hash": "da2808822e8cd8553c3b108bfa88e52756c187499fd42ee2c7f1fac9808ecf15", "id": "B2.9a", "kind": "int", "magnitude": 7, "mistakes": {"b3d2c93bf22ec48dd3e6d4023847d1439611eed5be9a571d1efac7520765410a": "une table de V lignes et de d colonnes : multiplie", "dc1a4b674d0698322f020b43c46a331f2d861b7a7014745970a66d6bac28eb1f": "GPT-2 lie ses poids : la tête de sortie est la même matrice que la table"}, "sign": 1}
📝 Ex B2.9a : réponse enregistrée (int).
WB_ANSWER {"decimals": 1, "hash": "5371b0b2ec33d3b6d41ef1c3c69de7b57fc6636ed41f8a32df12f6fe15068748", "hash_coarse": "6b6ca42b373e834eec160e14846717666ba398e860621b59a6fda041cc5a37c7", "id": "B2.9b", "kind": "float", "magnitude": 1, "mistakes": {"9d5de9b4eeae9453fb2bd11540172da8ad47d0bc75ca79d3c945c31a97f0a8de": "on demande un pourcentage, pas une proportion", "a91178476eabf241c9461a76dda319c1146d0bfce9bbfe0a4d8a49de7d754c67": "GPT-2 lie ses poids : la table n'est comptée qu'une fois"}, "sign": 1}
📝 Ex B2.9b : réponse enregistrée 

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu compter les octets et les points de code d'un texte, et écrire un caractère en UTF-8 ?
2. Sais-tu dérouler un entraînement BPE à la main, départage compris, puis encoder un mot nouveau par rang ?
3. Sais-tu dire quelles lignes d'une table d'embedding reçoivent un gradient, et calculer une PPMI ?
4. Sais-tu chiffrer ce que coûtent une table d'embedding et un texte selon sa langue ?

**Pour aller plus loin** : le chapitre 6 du cours LLM de Hugging Face et `minbpe` d'A. Karpathy, cités dans la fiche. La suite : le bonus B3, attention et Transformers.

*Les exercices de code B2.13 à B2.31 (parties A à E : tokenizers réels, ton BPE, expériences, vecteurs de mots, embeddings de phrases, livraison et défi) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py B2` pour obtenir le notebook complet.*